In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
!pip install -U langchain langchain-community langchain-huggingface langchain-core langchain-text-splitters chromadb sentence-transformers
!pip install -qU langchain-huggingface sentence-transformers accelerate bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 6.4 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of transformers to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.7/111.7 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 120.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 500.1/500.1 kB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.4/21.4 MB 111.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 110.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 78.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.1/158.

In [ ]:
import os
import shutil

# 경로 확인 (본인 경로로 수정 필수)
DB_PATH = "/content/drive/MyDrive/DILAB/YB/DILAB/LCK_RAG_Experiment/VectorDB"

if os.path.exists(DB_PATH):
    print(f"깨진 DB 청소 중: {DB_PATH}")
    try:
        shutil.rmtree(DB_PATH)
        print("삭제 완료.")
    except Exception as e:
        print(e)
else:
    print("이미 삭제되어 없습니다. 바로 생성 단계로 가세요.")

깨진 DB 청소 중: /content/drive/MyDrive/DILAB/YB/DILAB/LCK_RAG_Experiment/VectorDB
삭제 완료.


In [3]:
import json
import os
import shutil
import time
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# 1. 경로 설정
BASE_DIR = "/content/drive/MyDrive/DILAB/YB/DILAB/LCK_RAG_Experiment/data"
INPUT_FILE_PATH = os.path.join(BASE_DIR, "lck_final_complete.json")
DB_PATH = "/content/drive/MyDrive/DILAB/YB/DILAB/LCK_RAG_Experiment/VectorDB"

# 2. 데이터 로드
print(f"[데이터 로딩 시작]: {INPUT_FILE_PATH}")

if not os.path.exists(INPUT_FILE_PATH):
    print(f"경로 오류: {INPUT_FILE_PATH}")
else:
    with open(INPUT_FILE_PATH, "r", encoding="utf-8") as f:
        raw_data = json.load(f)

    documents = []
    for item in raw_data:
        text_content = f"제목: {item['title']}\n내용: {item['content']}"
        metadata = {
            "source": item["url"],
            "date": item["date"],
            "title": item["title"]
        }
        doc = Document(page_content=text_content, metadata=metadata)
        documents.append(doc)

    print(f"기사 로드 완료: 총 {len(documents)}개")

    # 3. 텍스트 청킹 (Chunking)
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=900,
        chunk_overlap=200,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"[청킹 완료]: 총 {len(split_docs)}개의 청크 생성")

    # 4. 임베딩 및 DB 저장
    print("[임베딩 모델 로딩 중...]")

    embedding_model = HuggingFaceEmbeddings(
        model_name="jhgan/ko-sroberta-multitask",
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True}
    )

    print(f"[백터 DB 저장 시작]: {DB_PATH}")

    # 기존 폴더 삭제 후 재생성
    if os.path.exists(DB_PATH):
        try:
            shutil.rmtree(DB_PATH)
            time.sleep(2)
        except OSError as e:
            print(e)

    os.makedirs(DB_PATH, exist_ok=True)

    try:
        # DB 생성 및 저장
        vectordb = Chroma.from_documents(
            documents=split_docs,
            embedding=embedding_model,
            persist_directory=DB_PATH
        )

        try:
            vectordb.persist()
        except:
            pass

        # 5. 데이터 검증
        print("\n실제로 데이터가 잘 들어갔는지 확인합니다...")

        # DB를 다시 불러와서 개수 세보기
        check_db = Chroma(persist_directory=DB_PATH, embedding_function=embedding_model)
        count = check_db._collection.count()

        print(f"최종 저장된 문서 개수: {count}개")

        if count > 0:
            print("검증 성공!")
        else:
            print("데이터가 0개입니다.")

    except Exception as e:
        print(e)

[데이터 로딩 시작]: /content/drive/MyDrive/DILAB/YB/DILAB/LCK_RAG_Experiment/data/lck_final_complete.json
기사 로드 완료: 총 266개
[청킹 완료]: 총 626개의 청크 생성
[임베딩 모델 로딩 중...]


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[백터 DB 저장 시작]: /content/drive/MyDrive/DILAB/YB/DILAB/LCK_RAG_Experiment/DB


/tmp/ipython-input-2812016288.py:77: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectordb.persist()
/tmp/ipython-input-2812016288.py:85: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  check_db = Chroma(persist_directory=DB_PATH, embedding_function=embedding_model)



실제로 데이터가 잘 들어갔는지 확인합니다...
최종 저장된 문서 개수: 626개
검증 성공!


In [4]:
import json
import os
import shutil
import time
import torch
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# 1. 경로 설정
BASE_DIR = "/content/drive/MyDrive/DILAB/YB/DILAB/LCK_RAG_Experiment/data"
INPUT_FILE_PATH = os.path.join(BASE_DIR, "lck_final_complete.json")
DB_PATH = "/content/drive/MyDrive/DILAB/YB/DILAB/LCK_RAG_Experiment/VectorDB"

# 2. 데이터 로드
print(f"[데이터 로딩 시작]: {INPUT_FILE_PATH}")

if not os.path.exists(INPUT_FILE_PATH):
    print(f"경로 오류: {INPUT_FILE_PATH}")
else:
    with open(INPUT_FILE_PATH, "r", encoding="utf-8") as f:
        raw_data = json.load(f)

    documents = []
    for item in raw_data:
        text_content = f"제목: {item['title']}\n내용: {item['content']}"
        metadata = {
            "source": item["url"],
            "date": item["date"],
            "title": item["title"]
        }
        doc = Document(page_content=text_content, metadata=metadata)
        documents.append(doc)

    print(f"기사 로드 완료: 총 {len(documents)}개")

    # 3. 텍스트 청킹 (Chunking)
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"[청킹 완료]: 총 {len(split_docs)}개의 청크 생성")

    # 4. 임베딩 및 DB 저장 (Octen-Embedding-8B 적용)
    print("[임베딩 모델 로딩 중...]")

    model_name = "Octen/Octen-Embedding-8B"

    # A100(40GB) 환경 설정
    model_kwargs = {
        "device": "cuda",
        "trust_remote_code": True, # 커스텀 모델 신뢰 설정
        "model_kwargs": {
            "torch_dtype": torch.float16,
            # "load_in_4bit": True
        }
    }

    # 인코딩 설정
    encode_kwargs = {
        "normalize_embeddings": True,
        "batch_size": 16
    }

    try:
        embedding_model = HuggingFaceEmbeddings(
            model_name=model_name,
            model_kwargs=model_kwargs,
            encode_kwargs=encode_kwargs
        )
    except Exception as e:
        print(f"모델 로드 중 에러 발생: {e}")
        exit()

    print(f"[백터 DB 저장 시작]: {DB_PATH}")

    if os.path.exists(DB_PATH):
        try:
            shutil.rmtree(DB_PATH)
            time.sleep(2)
        except OSError as e:
            print(e)

    os.makedirs(DB_PATH, exist_ok=True)

    try:
        # DB 생성 및 저장
        vectordb = Chroma.from_documents(
            documents=split_docs,
            embedding=embedding_model,
            persist_directory=DB_PATH
        )

        try:
            vectordb.persist()
        except:
            pass # 최신 버전에서는 메서드가 없을 수 있음

        # 5. 데이터 검증
        print("\n실제로 데이터가 잘 들어갔는지 확인합니다...")

        # 검증용 DB 로드 (임베딩 모델을 다시 넘겨줘야 함)
        check_db = Chroma(persist_directory=DB_PATH, embedding_function=embedding_model)
        count = check_db._collection.count()

        print(f"최종 저장된 문서 개수: {count}개")

        if count > 0:
            print("검증 성공!")
            print("테스트 검색: 'T1 경기 결과'")
            test_res = check_db.similarity_search("T1 경기 결과", k=1)
            print(f"검색 결과 샘플: {test_res[0].page_content[:50]}...")
        else:
            print("데이터가 0개입니다.")

    except Exception as e:
        print(f"DB 저장 중 에러 발생: {e}")

[데이터 로딩 시작]: /content/drive/MyDrive/DILAB/YB/DILAB/LCK_RAG_Experiment/data/lck_final_complete.json
기사 로드 완료: 총 266개
[청킹 완료]: 총 561개의 청크 생성
[임베딩 모델 로딩 중...]


modules.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/217 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/298 [00:00<?, ?B/s]

[백터 DB 저장 시작]: /content/drive/MyDrive/DILAB/YB/DILAB/LCK_RAG_Experiment/VectorDB

실제로 데이터가 잘 들어갔는지 확인합니다...
최종 저장된 문서 개수: 561개
검증 성공!
테스트 검색: 'T1 경기 결과'
검색 결과 샘플: 제목: [LCK컵] T1, 한화생명 상대로 짜릿한 패승승! 첫 승 신고...


In [ ]:
import json
import os
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# 1. 설정
MATCH_FILE = "/content/drive/MyDrive/DILAB/YB/DILAB/LCK_RAG_Experiment/data/match_results.json"
DB_PATH = "/content/drive/MyDrive/DILAB/YB/DILAB/LCK_RAG_Experiment/DB"

# 2. 경기 데이터 로드 및 문서 변환
if not os.path.exists(MATCH_FILE):
    print(f"{MATCH_FILE} 파일이 없습니다.")
else:
    with open(MATCH_FILE, "r", encoding="utf-8") as f:
        match_data = json.load(f)

    match_documents = []
    print(f"경기 데이터 {len(match_data)}개를 변환")

    for item in match_data:
        content = item.get("description", "")

        # 혹시 description이 비어있을 경우
        if not content:
            content = f"날짜: {item.get('date')} | 경기: {item.get('team_a')} vs {item.get('team_b')} | 스코어: {item.get('score')} | 상태: {item.get('status')}"

        # 메타데이터 설정 (필터링 용도)
        metadata = {
            "source": "Match_Result", # 뉴스랑 구분하기 위한 태그
            "type": "fact",
            "date": item.get("date"),
            "title": f"{item.get('team_a')} vs {item.get('team_b')} 경기 결과"
        }

        match_documents.append(Document(page_content=content, metadata=metadata))

    # 3. 기존 DB 불러오기 & 데이터 추가
    print("임베딩 모델 로드 중")
    embedding_model = HuggingFaceEmbeddings(
        model_name="jhgan/ko-sroberta-multitask",
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True}
    )

    if os.path.exists(DB_PATH):
        print(f"기존 DB 로드 중: {DB_PATH}")
        vectordb = Chroma(
            persist_directory=DB_PATH,
            embedding_function=embedding_model
        )

        print(f"경기 데이터 {len(match_documents)}개를 기존 DB에 추가하는 중...")
        vectordb.add_documents(match_documents)

        print("경기 결과 데이터가 DB에 추가되었습니다.")

        # 확인용
        print(f"현재 DB 저장된 총 문서 수: {vectordb._collection.count()}개")

    else:
        print(f"{DB_PATH}")

경기 데이터 24개를 변환
임베딩 모델 로드 중
기존 DB 로드 중: /content/drive/MyDrive/DILAB/YB/DILAB/LCK_RAG_Experiment/DB
경기 데이터 24개를 기존 DB에 추가하는 중...
경기 결과 데이터가 DB에 추가되었습니다.
현재 DB 저장된 총 문서 수: 575개
